In [ ]:
#%pip install --upgrade sagemaker sagemaker-core sagemaker-train

In [ ]:
#import sagemaker
#from sagemaker import get_execution_role
#import boto3
#import os
#from sagemaker.session import Session
#from sagemaker.feature_store.feature_group import FeatureGroup

#bucket = sagemaker.Session().default_bucket()
#prefix = 'sagemaker/DEMO-xgboost-dm'
# Define IAM role
#role = get_execution_role()

In [1]:
import boto3
import os
import pandas as pd

from sagemaker.core.helper.session_helper import (
    Session,
    get_execution_role
)
from sagemaker.core import image_uris

sess = Session()
region = sess.boto_region_name
bucket = sess.default_bucket()
role = get_execution_role(sagemaker_session=sess)

prefix = "sagemaker/DEMO-xgboost-dm"

print("Region:", region)
print("Bucket:", bucket)
print("Role:", role)

Region: us-east-2
Bucket: sagemaker-us-east-2-115891590640
Role: arn:aws:iam::115891590640:role/service-role/AmazonSageMaker-ExecutionRole-20261024T213182


In [2]:
# AWS session in SageMaker Studio
boto_session = boto3.Session()
region = boto_session.region_name

sm_client = boto_session.client("sagemaker")
sts_client = boto_session.client("sts")
s3_client = boto_session.client("s3")

# Inspect the AWS identity available to this notebook
identity = sts_client.get_caller_identity()

print("AWS region:", region)
print("AWS account:", identity["Account"])
print("Current AWS identity:", identity["Arn"])

# Project configuration
prefix = "sagemaker/DEMO-xgboost-dm"

# Use your existing SageMaker default bucket if known.
# Otherwise, get the account's default SageMaker bucket after
# the SageMaker session/import issue has been resolved.


AWS region: us-east-2
AWS account: 115891590640
Current AWS identity: arn:aws:sts::115891590640:assumed-role/AmazonSageMaker-ExecutionRole-20261024T213182/SageMaker


In [ ]:
region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)

sagemaker_client = boto_session.client(service_name='sagemaker', region_name=region)
featurestore_runtime = boto_session.client(service_name='sagemaker-featurestore-runtime', region_name=region)

feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime
)

feature_group_name = "FG-my-data-flow-d10008f1"  # replace with your feature group name 
feature_group = FeatureGroup(name=feature_group_name, sagemaker_session=feature_store_session)

In [ ]:
import boto3

from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.mlops.feature_store import FeatureGroup

# SageMaker SDK v3 session
sess = Session()

# AWS region and Studio execution role
region = sess.boto_region_name
role = get_execution_role()

# Default S3 bucket and project prefix
bucket = sess.default_bucket()
prefix = "sagemaker/DEMO-xgboost-dm"

# AWS clients
boto_session = boto3.Session(region_name=region)

sagemaker_client = boto_session.client("sagemaker")
featurestore_runtime = boto_session.client("sagemaker-featurestore-runtime")

# SageMaker session configured with the boto3 clients
feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)

# Existing Feature Store feature group
feature_group_name = "FG-my-data-flow-d10008f1"

feature_group = FeatureGroup(
    name=feature_group_name,
    sagemaker_session=feature_store_session,
)

print("Region:", region)
print("Execution role:", role)
print("S3 bucket:", bucket)
print("Feature group:", feature_group_name)
print("Setup completed.")


In [ ]:
# Build SQL query to features group
fs_query = feature_group.athena_query()
fs_table = fs_query.table_name
query_string = 'SELECT * FROM "'+fs_table+'"'
print('Running ' + query_string)

In [ ]:
# Run Athena query. The output is loaded to a Pandas dataframe.
fs_query.run(query_string=query_string, output_location='s3://'+bucket+'/'+prefix+'/fs_query_results/')
fs_query.wait()
model_data = fs_query.as_dataframe()

In [ ]:
print(model_data)

In [3]:
##processing manually since I cannot get datawrangler data through 


import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# --------------------------------------------------
# 1. Load the dataset
# --------------------------------------------------
df = pd.read_csv("storedata_total.csv")

# --------------------------------------------------
# 2. Convert Boolean fields
# --------------------------------------------------
boolean_cols = ["retained", "paperless", "refill", "doorstep"]

for col in boolean_cols:
    # Normalize values such as "TRUE", " false ", 1, and 0
    normalized = df[col].astype("string").str.strip().str.lower()

    df[col] = normalized.map({
        "1": True,
        "true": True,
        "yes": True,
        "0": False,
        "false": False,
        "no": False,
    })

# Invalid or missing Boolean values become NaN

# --------------------------------------------------
# 3. Process dates into year, month, and day
# --------------------------------------------------
date_cols = ["created", "firstorder", "lastorder"]

for col in date_cols:
    parsed = pd.to_datetime(df[col], errors="coerce")

    df[f"{col}_year"] = parsed.dt.year
    df[f"{col}_month"] = parsed.dt.month
    df[f"{col}_day"] = parsed.dt.day

# Remove the original date columns
df = df.drop(columns=date_cols)

# --------------------------------------------------
# 4. Ensure numeric columns contain numeric values
# --------------------------------------------------
numeric_cols = [
    "esent",
    "eopenrate",
    "eclickrate",
    "avgorder",
    "ordfreq",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# --------------------------------------------------
# 5. Drop rows with missing data
# --------------------------------------------------
# Includes missing values created by invalid dates,
# Boolean values, or nonnumeric values.
df = df.dropna().copy()

# Convert Boolean fields to consistent Boolean dtype
for col in boolean_cols:
    df[col] = df[col].astype(bool)

print("Rows remaining after cleaning:", len(df))

# --------------------------------------------------
# 6. Separate target and predictors
# --------------------------------------------------
# Predict retained; do not include it in X
y = df["retained"].astype(int)

# Remove the target and customer identifier from features
X = df.drop(columns=["retained", "custid"])

# --------------------------------------------------
# 7. Define feature groups
# --------------------------------------------------
categorical_cols = ["favday", "city"]

boolean_feature_cols = [
    "paperless",
    "refill",
    "doorstep",
]

date_feature_cols = [
    "created_year", "created_month", "created_day",
    "firstorder_year", "firstorder_month", "firstorder_day",
    "lastorder_year", "lastorder_month", "lastorder_day",
]

# --------------------------------------------------
# 8. Split data: 70% train, 20% evaluation, 10% validation
# --------------------------------------------------
X_train, X_remaining, y_train, y_remaining = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

X_eval, X_validation, y_eval, y_validation = train_test_split(
    X_remaining,
    y_remaining,
    test_size=1/3,
    random_state=42,
    stratify=y_remaining,
)

print(f"Training:   {len(X_train):,} rows ({len(X_train)/len(X)*100:.1f}%)")
print(f"Evaluation: {len(X_eval):,} rows ({len(X_eval)/len(X)*100:.1f}%)")
print(f"Validation: {len(X_validation):,} rows ({len(X_validation)/len(X)*100:.1f}%)")

# --------------------------------------------------
# 9. Build the preprocessing pipeline
# --------------------------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", "passthrough", numeric_cols),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            categorical_cols,
        ),
        ("boolean", "passthrough", boolean_feature_cols),
        ("dates", "passthrough", date_feature_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

# --------------------------------------------------
# 10. Fit on training data ONLY
# --------------------------------------------------
X_train_array = preprocessor.fit_transform(X_train)

# Transform other datasets without refitting
X_eval_array = preprocessor.transform(X_eval)
X_validation_array = preprocessor.transform(X_validation)

# --------------------------------------------------
# 11. Convert processed arrays back to DataFrames
# --------------------------------------------------
feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_array, columns=feature_names, index=X_train.index
)

X_eval_processed = pd.DataFrame(
    X_eval_array, columns=feature_names, index=X_eval.index
)

X_validation_processed = pd.DataFrame(
    X_validation_array,
    columns=feature_names,
    index=X_validation.index,
)

# --------------------------------------------------
# 12. Prepare datasets with target as first column
# --------------------------------------------------
# SageMaker built-in XGBoost expects label first,
# with no column headers in the final CSV files.

train_data = X_train_processed.copy()
train_data.insert(0, "retained", y_train)

eval_data = X_eval_processed.copy()
eval_data.insert(0, "retained", y_eval)

validation_data = X_validation_processed.copy()
validation_data.insert(0, "retained", y_validation)

# --------------------------------------------------
# 13. Save datasets to CSV
# --------------------------------------------------
train_data.to_csv("train.csv", index=False, header=False)
eval_data.to_csv("eval.csv", index=False, header=False)
validation_data.to_csv(
    "validation.csv", index=False, header=False
)

print("\nPreprocessing complete.")
print("Training dataset:", train_data.shape)
print("Evaluation dataset:", eval_data.shape)
print("Validation dataset:", validation_data.shape)
print("Saved train.csv, eval.csv, and validation.csv")


/tmp/ipykernel_3572/645620098.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[col], errors="coerce")
/tmp/ipykernel_3572/645620098.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[col], errors="coerce")
/tmp/ipykernel_3572/645620098.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[col], errors="coerce")


Rows remaining after cleaning: 30758
Training:   21,530 rows (70.0%)
Evaluation: 6,152 rows (20.0%)
Validation: 3,076 rows (10.0%)

Preprocessing complete.
Training dataset: (21530, 29)
Evaluation dataset: (6152, 29)
Validation dataset: (3076, 29)
Saved train.csv, eval.csv, and validation.csv


In [4]:
# Assumes bucket and prefix are already defined

s3 = boto3.client("s3", region_name="us-east-2")

# Upload each split to its own S3 folder
uploads = {
    "train.csv": f"{prefix}/train/train.csv",
    "eval.csv": f"{prefix}/eval/eval.csv",
    "validation.csv": f"{prefix}/validation/validation.csv",
}

for local_file, s3_key in uploads.items():
    s3.upload_file(local_file, bucket, s3_key)
    print(f"Uploaded s3://{bucket}/{s3_key}")

print("\nAll datasets uploaded.")

Uploaded s3://sagemaker-us-east-2-115891590640/sagemaker/DEMO-xgboost-dm/train/train.csv
Uploaded s3://sagemaker-us-east-2-115891590640/sagemaker/DEMO-xgboost-dm/eval/eval.csv
Uploaded s3://sagemaker-us-east-2-115891590640/sagemaker/DEMO-xgboost-dm/validation/validation.csv

All datasets uploaded.


In [ ]:
#container = sagemaker.image_uris.retrieve(region=boto3.Session().region_name, framework='xgboost', version='latest')

In [5]:
from sagemaker.core import image_uris

region = boto3.Session().region_name

container = image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1",
)

print("XGBoost container:", container)

XGBoost container: 257758044811.dkr.ecr.us-east-2.amazonaws.com/sagemaker-xgboost:1.7-1


In [ ]:
#s3_input_train = sagemaker.inputs.TrainingInput(s3_data='s3://{}/{}/train'.format(bucket, prefix), content_type='csv')
#s3_input_validation = sagemaker.inputs.TrainingInput(s3_data='s3://{}/{}/validation/'.format(bucket, prefix), content_type='csv')

In [7]:
from sagemaker.train.configs import InputData

s3_input_train = InputData(
    channel_name="train",
    data_source=f"s3://{bucket}/{prefix}/train/"
)

s3_input_eval = InputData(
    channel_name="validation",
    data_source=f"s3://{bucket}/{prefix}/eval/"
)

s3_input_final_validation = InputData(
    channel_name="final_validation",
    data_source=f"s3://{bucket}/{prefix}/validation/"
)

/tmp/ipykernel_3572/3149762410.py:1: DeprecationWarning: sagemaker.train.configs has been moved to sagemaker.core.training.configs. Please update your imports. This shim will be removed in a future version.
  from sagemaker.train.configs import InputData


In [12]:
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import Compute, InputData, OutputDataConfig
from sagemaker.core.helper.session_helper import (
    Session,
    get_execution_role,
)
from sagemaker.core import image_uris

# 1. SageMaker configuration
sess = Session()
region = sess.boto_region_name
role = get_execution_role(sagemaker_session=sess)

output_path = f"s3://{bucket}/{prefix}/output/"

container = image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1",
)

# 2. XGBoost hyperparameters
hyperparameters = {
    "max_depth": "5",
    "eta": "0.2",
    "gamma": "4",
    "min_child_weight": "6",
    "subsample": "0.8",
    "verbosity": "1",
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "num_round": "100",
}

# 3. Compute configuration
compute = Compute(
    instance_type="ml.m4.xlarge",
    instance_count=1,
    volume_size_in_gb=10,
)

# 4. Input channels
s3_input_train = InputData(
    channel_name="train",
    data_source=f"s3://{bucket}/{prefix}/train/",
    content_type="text/csv",
)

s3_input_eval = InputData(
    channel_name="validation",
    data_source=f"s3://{bucket}/{prefix}/eval/",
    content_type="text/csv",
)

# 5. Configure trainer
xgb = ModelTrainer(
    training_image=container,
    role=role_arn,
    compute=compute,
    hyperparameters=hyperparameters,
    base_job_name="retained-xgboost",
    output_data_config=OutputDataConfig(
        s3_output_path=output_path
    ),
)

# 6. Launch training
training_job = xgb.train(
    input_data_config=[
        s3_input_train,
        s3_input_eval,
    ],
    wait=True,
    logs=True,
)

print("Training job submitted/completed.")

RoleValidationError: IAM role 'arn:aws:iam::115891590640:role/SageMaker-AutoRole-Training' cannot be used for 'training' workloads.
Missing permissions: cloudwatch:PutMetricData, ec2:CreateNetworkInterface, ec2:CreateNetworkInterfacePermission, ec2:DeleteNetworkInterface, ec2:DeleteNetworkInterfacePermission, ec2:DescribeDhcpOptions, ec2:DescribeNetworkInterfaces, ec2:DescribeSecurityGroups, ec2:DescribeSubnets, ec2:DescribeVpcs, ecr:GetAuthorizationToken

To fix this, either:
  1. Grant the permissions above (and ensure 'sagemaker.amazonaws.com' is a trusted principal) on a role, then pass it via the interface's 'role'/'role_arn' parameter. If you cannot modify or create IAM roles, ask your administrator to do this; or
  2. Create a dedicated least-privilege role and pass its ARN:
       from sagemaker.core.helper import IamRoleResolver
       role_arn = IamRoleResolver().create_execution_role(role_type='training')

In [9]:
from sagemaker.core.helper import IamRoleResolver

resolver = IamRoleResolver(sagemaker_session=sess)

role_arn = resolver.create_execution_role(
    role_type="training",
    s3_resource=bucket,
)

print("Training role:", role_arn)

Training role: arn:aws:iam::115891590640:role/SageMaker-AutoRole-Training
